# DSCI 6883 - Descriptive Analytics & Data Integrity
## Discussion 1.6 Follow-Up: The Parts That Did Not Land

**Fall 2026 | Module 1 | Fernando Rubio Garcia**

In [0]:
get_ipython().kernel._abort_queues = lambda *a, **kw: None

### Where this came from

In the 1.6 discussion I asked you to point at the exact places where the Python Foundations course lost you. You did, and precise questions make precise answers possible: every section below answers one of your posts.

If your part is in here and it still does not click, reply in the discussion with the **section number and the cell** where it stopped making sense.

### How to use this notebook

1. **Run every cell yourself**, top to bottom. Use `Shift + Enter`.
2. **Predict the output before you run a cell.** Two of you wrote some version of "I can follow the examples, but I could not write it myself." Predicting is the step between those two.
3. **Do the `Your Turn` exercises.** Each one has a solution you can expand - try it first.
4. Some cells are **designed to fail**. They are marked `SUPPOSED TO FAIL`. Read the error message, then keep going.
5. If the notebook starts behaving strangely, use **Kernel > Restart Kernel and Run All Cells**. Section 5 explains what that actually does, and why it works.

### Outline

| Section | Topic | From the post by |
|---|---|---|
| 1 | Which variables a function can see (scope) | Tyler |
| 2 | List comprehensions: from reading them to writing them | Tyler |
| 3 | `enumerate`, dictionaries, dictionary comprehensions, and `import numpy as np` | Jacob |
| 4 | `lambda` and `map()` | Matei |
| 5 | The Kernel menu: what each option does and when to use it | Kara |
| 6 | Saving, finding, and importing your own modules (and what happened to Kara's tax calculator) | Matei, Kara |

> **Section 6 creates a few small files** whose names start with `demo_` in the same folder as this notebook. The last cell deletes them.

> **Working in Databricks, like Scott?** Sections 1-4 are plain Python. Sections 5 and 6 are specifically about how Jupyter handles memory and files on your own computer, so do those two in Jupyter.

---
# 1. Which variables can a function see? (Scope)

**From Tyler's post:** You can create a function and use a variable inside it, but it is not always clear which variables a function can reach and which it cannot.

Every time a function runs, Python gives it a private scratch space. Names created inside the function, including its parameters, live in that scratch space and are thrown away when the function returns. Names created outside every function, at the top level of the notebook, are **global**.

When Python meets a name inside a function, it searches in this order and stops at the first match:

| Order | Scope | Where the name was created |
|---|---|---|
| 1 | **L**ocal | inside this function, including its parameters |
| 2 | **E**nclosing | inside a function that contains this one (rare for now) |
| 3 | **G**lobal | at the top level of the notebook or module |
| 4 | **B**uilt-in | names Python always has: `print`, `len`, `sum`, ... |

That search order is called the **LEGB rule**. Three rules cover almost everything you will run into:

1. **A function can read a global variable.**
2. **If a function assigns to a name anywhere in its body (`=`, `+=`), that name is local for the whole function.** A global with the same name is not touched.
3. **Code outside a function cannot see the function's local variables.** Values go in through parameters and come out through `return`.

### 1.1 Local variables disappear when the function returns

In [0]:
def make_greeting(name):
    message = f"Hello, {name}!"    # message is local to make_greeting
    return message

result = make_greeting("Fernando")
print(result)

`result` exists because we stored what the function **returned**. `message` was the function's scratch work.

In [0]:
# SUPPOSED TO FAIL -- NameError
print(message)

The same is true for the parameter `name`: it only exists while `make_greeting` is running.

### 1.2 Reading a global works (Rule 1)

In [0]:
sales_tax = 0.08    # global: created outside any function

def price_with_tax(price):
    return round(price * (1 + sales_tax), 2)    # reads the global sales_tax

print(price_with_tax(50))

Python looks `sales_tax` up when the function **runs**, not when it is defined. **Predict** what happens if we change the global and call the function again:

In [0]:
sales_tax = 0.10
print(price_with_tax(50))

### 1.3 Assigning inside a function creates a new local (Rule 2)

**Predict** what the two `print` lines show before you run the cell.

In [0]:
counter = 0

def set_counter():
    counter = 99    # assignment -> a NEW local variable that happens to be named counter
    print("inside the function: ", counter)

set_counter()
print("outside the function:", counter)

Two different variables that share a name. The local one vanished when the function returned; the global one was never touched.

### 1.4 The confusing one: `UnboundLocalError`

This is the error that makes scope feel random. Read the function: it looks like it should add 1 to the global `counter`.

In [0]:
counter = 0

def add_one():
    counter = counter + 1    # counter is assigned in this function, so it is local everywhere in it
    return counter

In [0]:
# SUPPOSED TO FAIL -- UnboundLocalError
add_one()

Python decides which names are local when it **reads the function definition**, before any line runs. It sees `counter = ...` in the body, so `counter` is local for the entire function, including the right-hand side `counter + 1`. When that line runs, the local `counter` does not have a value yet, and Python does not fall back to the global.

The fix is almost never "make it global." It is: **pass the value in, return the new value out.**

In [0]:
# Best fix: parameter in, return value out
def add_one(value):
    return value + 1

counter = 0
counter = add_one(counter)
counter = add_one(counter)
print(counter)

There is also a keyword that tells Python "inside this function, this name means the global one." You will see it in other people's code:

In [0]:
# Works, but use sparingly
counter = 0

def add_one_global():
    global counter          # counter now refers to the global counter
    counter = counter + 1

add_one_global() # = 1
add_one_global() # = 2
print(counter)

Why prefer the first version? With a parameter and `return`, everything the function depends on is visible in the call `add_one(counter)`. With `global`, the function secretly depends on (and changes) a variable defined somewhere else in the notebook. That is exactly the kind of code that breaks after a kernel restart or when you copy the function into another notebook.

### 1.5 The list surprise: changing an object is not assigning a name

Rule 2 is triggered by **assigning to the name**. Calling a method that changes a list or dictionary in place is not an assignment, so it changes the global object, no `global` keyword needed.

In [0]:
shopping = ["eggs"]

def add_item(item):
    shopping.append(item)    # no `shopping = ...` in this function, so shopping is the global list

add_item("milk")
print(shopping)

In [0]:
def replace_list():
    shopping = ["bread"]     # assignment -> brand-new local list; the global list is untouched

replace_list()
print(shopping)

| Inside a function, without `global` | Changes the global? | Why |
|---|---|---|
| `shopping.append("milk")` | Yes | changes the list object; no name is assigned |
| `shopping[0] = "tea"` | Yes | changes an item inside the list; the name `shopping` is not reassigned |
| `inventory["pens"] = 12` | Yes | same idea, for a dictionary |
| `shopping = ["bread"]` | No | assigns the name, so a new local variable is created |
| `shopping += ["bread"]` | Error | `+=` counts as assigning, so `shopping` is local and has no value yet (`UnboundLocalError`) |

### 1.6 Built-ins are the last place Python looks

`sum`, `len`, `max`, `list`, and `print` are built-in names. A variable with the same name is found **first** and hides the built-in.

In [0]:
def total_points(points):
    sum = 0                   # a local named sum hides the built-in sum() inside this function
    return sum(points)

In [0]:
# SUPPOSED TO FAIL -- TypeError
total_points([10, 20, 30])

`'int' object is not callable` means "you put parentheses after something that is a number, not a function."

In a notebook this usually happens at the top level: run `sum = 0` or `list = [1, 2, 3]` in any cell, and the built-in stays hidden **in every cell** until you restart the kernel (or run `del sum`). Pick names like `total` and `values` instead.

### 1.7 Why scope bugs hide in notebooks

Every variable you create at the top level of a notebook is global, and it stays in memory until the kernel restarts, even if you delete the cell that created it. So a function can quietly depend on a global you never meant to use.

In [0]:
def class_average():
    return sum(scores) / len(scores)    # scores is not a parameter...

scores = [88, 92, 79]
print(class_average())                  # ...but this works, because a global named scores exists

In [0]:
del scores

Now simulate a fresh kernel, or pasting that function into another notebook, where `scores` was never created:

In [0]:
# SUPPOSED TO FAIL -- NameError
del scores
print(class_average())

The fix is the same as in 1.4: make the input a parameter, so the function works no matter what else is (or is not) in memory.

In [0]:
def class_average(scores):
    return sum(scores) / len(scores)

print(class_average([88, 92, 79]))
print(class_average([70, 75]))

#### Your Turn 1.1 - Predict, then run

Write down what the three `print` lines will show. Then run the cell and compare.

In [0]:
level = "global"

def show_level():
    print("1:", level)

def change_level():
    level = "local"
    print("2:", level)

show_level()
change_level()
print("3:", level)

<details>
<summary><b>Show answer</b></summary>

```
1: global
2: local
3: global
```

- `show_level` only **reads** `level`, so it finds the global (Rule 1).
- `change_level` **assigns** `level`, so inside it `level` is a new local (Rule 2).
- The global was never changed, so line 3 still shows `global`.

</details>

#### Your Turn 1.2 - Fix the function

This function raises `UnboundLocalError`:

```python
total = 0

def add_to_total(amount):
    total = total + amount
```

Rewrite it **without** `global` so that this code prints `15`:

```python
total = 0
total = add_to_total(total, 10)
total = add_to_total(total, 5)
print(total)
```

In [0]:
# Your code here.

<details>
<summary><b>Show answer</b></summary>

```python
def add_to_total(total, amount):
    return total + amount

total = 0
total = add_to_total(total, 10)
total = add_to_total(total, 5)
print(total)    # 15
```

The function no longer needs to know anything about the notebook. It receives the current total, returns the new one, and the caller decides where to store it.

</details>

#### Your Turn 1.3 - Which calls change the list?

Predict what `print(names)` shows, then run the cell. Use the table in 1.5 to explain each function.

In [0]:
names = ["Ana"]

def a():
    names.append("Ben")

def b():
    names = ["Chloe"]

def c():
    names[0] = "Dana"

a()
b()
c()
print(names)

<details>
<summary><b>Show answer</b></summary>

`['Dana', 'Ben']`

- `a()` changes the global list in place, so it becomes `['Ana', 'Ben']`.
- `b()` assigns the name `names`, which creates a local list inside `b`. The global list is untouched.
- `c()` replaces an **item** of the global list without assigning the name `names`, so it becomes `['Dana', 'Ben']`.

</details>

---
# 2. List comprehensions: from reading them to writing them

**From Tyler's post:** You can read a list comprehension, but writing one takes longer than writing a normal loop, and side-by-side comparisons with loops would help.

Good news first: **you never have to write a comprehension from scratch.** Write the loop you already know how to write, then translate it. The translation is mechanical, and after doing it twenty or so times you will start skipping the loop on your own.

A comprehension is a shortcut for exactly one loop shape: **start with an empty list, loop, maybe check a condition, append.**

```python
new_list = []
for ITEM in ITERABLE:
    if CONDITION:
        new_list.append(EXPRESSION)
```

becomes

```python
new_list = [EXPRESSION for ITEM in ITERABLE if CONDITION]
```

**The recipe:**

1. Write `[ ]`.
2. Inside it, first write what was inside `.append( )`.
3. Then copy the `for` line, without the colon.
4. Then copy the `if` line, without the colon (if there is one).

What you **keep** goes first. The `for` and `if` parts follow in the same top-to-bottom order they had in the loop.

### 2.1 Transform every item

In each cell below, the loop and the comprehension build the same list, and the last line checks that they match.

In [0]:
numbers = [1, 2, 3, 4, 5, 6]

# Loop
squares = []
for n in numbers:
    squares.append(n ** 2)

# Comprehension: what was inside .append( ), then the for line
squares_comp = [n ** 2 for n in numbers]

print(squares_comp)
print(squares == squares_comp)

### 2.2 Keep only some items (filter)

In [0]:
numbers = [1, 2, 3, 4, 5, 6]

# Loop
evens = []
for n in numbers:
    if n % 2 == 0:
        evens.append(n)

# Comprehension: the append part, then the for line, then the if line
evens_comp = [n for n in numbers if n % 2 == 0]

print(evens_comp)
print(evens == evens_comp)

When you keep each item unchanged, the front part is just the loop variable: `n for n in ...`. It looks repetitive, but the two `n`s have different jobs. The first one is "what to keep"; the second one names each item as the loop walks through the list.

### 2.3 Filter and transform together

In [0]:
names = ["al", "beatrice", "cy", "dominic", "eve"]

# Loop
long_names = []
for name in names:
    if len(name) > 3:
        long_names.append(name.title())

# Comprehension
long_names_comp = [name.title() for name in names if len(name) > 3]

print(long_names_comp)
print(long_names == long_names_comp)

### 2.4 `if`/`else`: the one that trips everyone up

A filter `if` (no `else`) decides **whether** an item is kept, so it goes at the **end**.

An `if`/`else` decides **what** to keep for every item, so it is part of the expression and goes at the **front**.

In [0]:
scores = [95, 62, 78, 49, 88]

# Loop: every score produces a label, so BOTH branches append
labels = []
for s in scores:
    if s >= 70:
        labels.append("pass")
    else:
        labels.append("fail")

# Comprehension: the if/else collapses into ONE expression, which takes the append position
labels_comp = ["pass" if s >= 70 else "fail" for s in scores]

print(labels_comp)
print(labels == labels_comp)

Read `"pass" if s >= 70 else "fail"` as a single value: *"pass" when the score is at least 70, otherwise "fail"*. That whole value is what gets appended, so it goes where the append part always goes.

Putting an `if`/`else` at the end is a syntax error:

In [0]:
# SUPPOSED TO FAIL -- SyntaxError
labels_comp = [s for s in scores if s >= 70 else "fail"]

| You want to... | Where the `if` goes | Example |
|---|---|---|
| keep **some** of the items | at the end, no `else` | `[s for s in scores if s >= 70]` |
| keep **every** item, choosing between two values | at the front, with `else` | `["pass" if s >= 70 else "fail" for s in scores]` |

### 2.5 Looping over pairs: `zip` and `enumerate`

Anything you can write after `for` in a loop, you can write after `for` in a comprehension, including two names at once.

In [0]:
unit_prices = [2.50, 10.00, 4.25]
quantities = [4, 1, 3]

# Loop
line_totals = []
for price, qty in zip(unit_prices, quantities):
    line_totals.append(price * qty)

# Comprehension
line_totals_comp = [price * qty for price, qty in zip(unit_prices, quantities)]

print(line_totals_comp)
print(line_totals == line_totals_comp)

In [0]:
runners = ["Kim", "Luis", "Maya"]

# Loop
podium = []
for place, runner in enumerate(runners, start=1):
    podium.append(f"{place}. {runner}")

# Comprehension
podium_comp = [f"{place}. {runner}" for place, runner in enumerate(runners, start=1)]

print(podium_comp)
print(podium == podium_comp)

### 2.6 Nested loops

With two `for` lines, copy them **in the same order** they appear in the loop: the outer loop first.

In [0]:
hours_by_week = [[3, 5, 2], [4, 6], [1, 7, 8]]

# Loop
all_hours = []
for week in hours_by_week:
    for h in week:
        all_hours.append(h)

# Comprehension: the append part, then the outer for, then the inner for
all_hours_comp = [h for week in hours_by_week for h in week]

print(all_hours_comp)
print(all_hours == all_hours_comp)

### 2.7 When to keep the loop

A comprehension only fits the "empty list, loop, maybe `if`, append" shape. Keep the regular loop when:

- each step depends on the previous one (running totals, "the best so far"),
- you need `break`, or several statements for each item,
- you are doing something for its effect, like `print`. `[print(x) for x in items]` runs, but it builds a useless list of `None` values.

In [0]:
deposits = [100, 250, 75]

# Each balance depends on the previous one, so a regular loop is the clear choice
balances = []
balance = 0
for amount in deposits:
    balance = balance + amount
    balances.append(balance)

print(balances)

One difference besides the syntax, and it connects back to Section 1: **a comprehension has its own scope.** The variable of a regular `for` loop still exists after the loop ends. The variable inside a comprehension does not exist outside the brackets.

In [0]:
for loop_item in [10, 20, 30]:
    pass

print(loop_item)    # the for-loop variable survives the loop

In [0]:
# SUPPOSED TO FAIL -- NameError
doubled = [comp_item * 2 for comp_item in [10, 20, 30]]
print(comp_item)

#### Your Turn 2.1 - Translate the loops

Each cell below has a working loop. Replace the empty list `[]` on the `_comp` line with a comprehension, and re-run the cell until the check prints `True`. Use the recipe: the append part first, then the `for`, then the `if`.

In [0]:
# a) The length of every word
words = ["data", "integrity", "audit", "py"]

lengths = []
for w in words:
    lengths.append(len(w))

lengths_comp = []    # your comprehension here
print(lengths_comp == lengths)

In [0]:
# b) Only the prices under 20
item_prices = [12.99, 45.00, 5.50, 19.99, 20.00]

cheap = []
for p in item_prices:
    if p < 20:
        cheap.append(p)

cheap_comp = []    # your comprehension here
print(cheap_comp == cheap)

In [0]:
# c) 10% off anything over 50; everything else unchanged
item_prices = [80.0, 35.0, 120.0, 50.0]

sale = []
for p in item_prices:
    if p > 50:
        sale.append(p * 0.9)
    else:
        sale.append(p)

sale_comp = []    # your comprehension here
print(sale_comp == sale)

In [0]:
# d) The capitalized first letter of every name that is not blank
signups = ["ana", "", "ben", "", "chloe"]

initials = []
for name in signups:
    if name != "":
        initials.append(name[0].upper())

initials_comp = []    # your comprehension here
print(initials_comp == initials)

**e) No loop this time.** Build a list of every multiple of 3 from 1 to 30. If you get stuck, write the loop first, then translate it.

In [0]:
# Your code here.

**f) Now go backwards.** Rewrite this comprehension as a regular loop that builds the same list:

```python
emails = [" ANA@EXAMPLE.COM", "Ben@Example.com ", "not an email"]
clean = [e.strip().lower() for e in emails if "@" in e]
```

In [0]:
# Your code here.

<details>
<summary><b>Show answer</b></summary>

```python
# a)
lengths_comp = [len(w) for w in words]

# b)
cheap_comp = [p for p in item_prices if p < 20]

# c) every price produces a value, so the if/else goes at the FRONT
sale_comp = [p * 0.9 if p > 50 else p for p in item_prices]

# d) only some names are kept, so the if goes at the END
initials_comp = [name[0].upper() for name in signups if name != ""]

# e)
multiples_of_3 = [n for n in range(1, 31) if n % 3 == 0]

# f)
emails = [" ANA@EXAMPLE.COM", "Ben@Example.com ", "not an email"]
clean = []
for e in emails:
    if "@" in e:
        clean.append(e.strip().lower())
print(clean)    # ['ana@example.com', 'ben@example.com']
```

In (b), notice that `20.00` is **not** kept: the condition is `p < 20`, not `p <= 20`.

</details>

---
# 3. `enumerate`, dictionaries, and dictionary comprehensions

**From Jacob's post:** `enumerate` took extra study, dictionaries in general are a struggle, and right now it is dictionary comprehensions and the `numpy` import.

These are more connected than they look. `enumerate`, `zip`, and a dictionary's `.items()` all do the same job: they hand a `for` loop **pairs**, and you unpack each pair into two names. Once that pattern is solid, a dictionary comprehension is Section 2's recipe with one small change.

### 3.1 `enumerate` hands you (position, item) pairs

In [0]:
fruits = ["apple", "banana", "cherry"]

list(enumerate(fruits))

That is all `enumerate` does: it pairs each item with its position. A `for` loop can then unpack each pair into two names. These two loops do exactly the same thing:

In [0]:
# Without unpacking: each pair arrives as one tuple, and we split it ourselves
for pair in enumerate(fruits):
    position, fruit = pair
    print(position, fruit)

print("---")

# With unpacking: the two names go straight into the for line
for position, fruit in enumerate(fruits):
    print(position, fruit)

`start=1` changes where the counting begins, which is handy for anything a person will read:

In [0]:
for rank, fruit in enumerate(fruits, start=1):
    print(f"{rank}. {fruit}")

`enumerate` replaces the older, clunkier way of getting positions:

```python
for i in range(len(fruits)):
    fruit = fruits[i]
```

The most common mistake is writing two names but forgetting `enumerate`. Python then tries to split each **string** into two pieces:

In [0]:
# SUPPOSED TO FAIL -- ValueError
for position, fruit in fruits:
    print(position, fruit)

`too many values to unpack (expected 2)` means "you gave me two names, but each item has more than two pieces." When you see it on a `for` line, check whether you forgot `enumerate(...)`, `zip(...)`, or `.items()`.

### 3.2 Dictionaries: look things up by a label instead of a position

A list answers "what is in position 2?" A dictionary answers "what goes with `"banana"`?" You choose the labels, called **keys**, and each key points to a **value**.

Think of the contacts on your phone: you look people up by name, not by "the 37th contact."

In [0]:
prices = {"apple": 0.50, "banana": 0.30, "cherry": 3.00}

print(prices["banana"])

Adding and updating use the same syntax. If the key already exists, its value is replaced. If not, the key is added.

In [0]:
prices["date"] = 2.00      # new key -> added
prices["apple"] = 0.60     # existing key -> updated

print(prices)

Asking for a key that is not there raises a `KeyError`:

In [0]:
# SUPPOSED TO FAIL -- KeyError
prices["mango"]

In [0]:
# Two safe ways to ask
print("mango" in prices)         # `in` checks the KEYS
print(prices.get("mango"))       # .get returns None when the key is missing...
print(prices.get("mango", 0))    # ...or the default you give it

### 3.3 Looping over a dictionary

Looping over a dictionary directly gives you only its **keys**. To get keys and values together, use `.items()`, which hands the loop pairs, exactly like `enumerate` did.

In [0]:
prices

In [0]:
for fruit in prices:
    print(fruit)

In [0]:
print(list(prices.items()))    # look familiar? pairs, just like enumerate

for fruit, price in prices.items():
    print(f"{fruit}: ${price:.2f}")

| To loop over... | Write | Each step gives you |
|---|---|---|
| a list, with positions | `for i, item in enumerate(my_list):` | position, item |
| two lists side by side | `for a, b in zip(list_a, list_b):` | one item from each list |
| a dictionary's keys | `for key in my_dict:` | key |
| a dictionary's keys and values | `for key, value in my_dict.items():` | key, value |

One pattern worth memorizing, because it shows up constantly in data work: **counting with a dictionary.**

In [0]:
votes = ["red", "blue", "red", "green", "blue", "red"]

counts = {}
for color in votes:
    counts[color] = counts.get(color, 0) + 1    # current count (or 0 if new), plus one, stored back

print(counts)

Read the middle line from the inside out: `counts.get(color, 0)` gets the current count, or `0` the first time a color shows up. Add `1`. Store the result back under the same key.

### 3.4 Dictionary comprehensions: Section 2's recipe, with `key: value`

The loop shape is the same as in Section 2, except that you **store under a key** instead of appending:

```python
new_dict = {}
for ITEM in ITERABLE:
    if CONDITION:
        new_dict[KEY] = VALUE
```

becomes

```python
new_dict = {KEY: VALUE for ITEM in ITERABLE if CONDITION}
```

Two changes from a list comprehension: **curly braces**, and the front part is **`key: value`** instead of a single expression. Everything after it (`for`, `if`) works exactly as in Section 2.

In [0]:
fruits = ["apple", "banana", "cherry"]

# Loop
name_lengths = {}
for fruit in fruits:
    name_lengths[fruit] = len(fruit)

# Comprehension: KEY: VALUE, then the for line
name_lengths_comp = {fruit: len(fruit) for fruit in fruits}

print(name_lengths_comp)
print(name_lengths == name_lengths_comp)

**From two lists** (keys in one, values in the other), use `zip`:

In [0]:
students = ["Ana", "Ben", "Chloe"]
grades = ["A", "B+", "A-"]

# Loop
gradebook = {}
for student, grade in zip(students, grades):
    gradebook[student] = grade

# Comprehension
gradebook_comp = {student: grade for student, grade in zip(students, grades)}

print(gradebook_comp)
print(gradebook == gradebook_comp)

**From an existing dictionary**, loop over `.items()` so you have both the key and the value to work with. This one raises every price by 10%:

In [0]:
# Loop
new_prices = {}
for fruit, price in prices.items():
    new_prices[fruit] = round(price * 1.10, 2)

# Comprehension
new_prices_comp = {fruit: round(price * 1.10, 2) for fruit, price in prices.items()}

print(new_prices_comp)
print(new_prices == new_prices_comp)

**Filtering a dictionary:** keep only the pairs you want by adding an `if` at the end.

In [0]:
cheap_fruit = {fruit: price for fruit, price in prices.items() if price < 1}

print(cheap_fruit)

**With `enumerate`:** use the position as the key, or the item as the key.

In [0]:
by_position = {i: fruit for i, fruit in enumerate(fruits, start=1)}
position_of = {fruit: i for i, fruit in enumerate(fruits, start=1)}

print(by_position)
print(position_of)
print(position_of["cherry"])

Forgetting `.items()` is the dictionary version of the mistake from 3.1. Looping over a dictionary gives only the keys, so Python tries to split each key **string** into two names:

In [0]:
# SUPPOSED TO FAIL -- ValueError
{fruit: price * 2 for fruit, price in prices}

One more trap: leave out the colon and you get a **set**, not a dictionary.

In [0]:
print(type({fruit: len(fruit) for fruit in fruits}))    # key: value -> dict
print(type({len(fruit) for fruit in fruits}))           # no colon   -> set

### 3.5 `import numpy as np`, and using NumPy inside a comprehension

`import numpy as np` is two ideas in one line:

- `import numpy` loads the NumPy package, so its tools are available as `numpy.mean(...)`, `numpy.max(...)`, and so on.
- `as np` gives the package a nickname. `np` **is** `numpy`, just shorter to type. Every NumPy example you will ever read uses `np`, which is the main reason to use it too.

The import styles side by side (Section 6 explains what `import` does behind the scenes):

| Line | How you call things |
|---|---|
| `import numpy` | `numpy.mean(values)` |
| `import numpy as np` | `np.mean(values)` |
| `from numpy import mean` | `mean(values)` |
| `from numpy import *` | **Avoid.** It dumps hundreds of NumPy names into your notebook, including NumPy's own `sum`, `any`, and `all`, which silently replace Python's built-ins (remember 1.6). |

If `import numpy as np` gives `ModuleNotFoundError: No module named 'numpy'`, NumPy is not installed in the Python your notebook is running. Run `%pip install numpy` in a cell, then run the import again. If it still fails, restart the kernel (Section 5) and try once more.

In [0]:
import numpy as np

quiz_scores = {
    "Ana":   [88, 92, 79, 95],
    "Ben":   [72, 65, 80, 70],
    "Chloe": [90, 94, 98, 91],
}

# Loop
averages = {}
for name, scores in quiz_scores.items():
    averages[name] = np.mean(scores)

# Comprehension: KEY: VALUE, then the for line over .items()
averages_comp = {name: np.mean(scores) for name, scores in quiz_scores.items()}

print(averages_comp)
print(averages == averages_comp)

Read it with the recipe: for each `name, scores` pair in `quiz_scores.items()`, the key is `name` and the value is `np.mean(scores)`.

**If your output shows `np.float64(88.5)` instead of `88.5`, nothing is wrong.** Starting with NumPy 2, NumPy labels its numbers with their type when they are displayed inside a dictionary or a list (a plain `print(np.mean(scores))` still shows `88.5`). It is still 88.5, and it does math like 88.5. For a cleaner display, convert each value with `float()`:

In [0]:
averages = {name: float(np.mean(scores)) for name, scores in quiz_scores.items()}

print(averages)

Once you have a dictionary, a second comprehension can filter it, and the value part can call any function you like:

In [0]:
honor_roll = {name: avg for name, avg in averages.items() if avg >= 85}
print(honor_roll)

highest = {name: int(np.max(scores)) for name, scores in quiz_scores.items()}
print(highest)

#### Your Turn 3.1 - Dictionary comprehensions

Run this cell first. Use its data (and `quiz_scores` from above) for all four parts.

In [0]:
inventory = {"pens": 12, "notebooks": 0, "markers": 5, "erasers": 0}
names = ["Ana", "Ben", "Chloe"]
emails = ["ana@example.com", "ben@example.com", "chloe@example.com"]
temps_f = {"Mon": 68, "Tue": 75, "Wed": 59}

a) Build `in_stock`: only the items in `inventory` with a count above 0.

b) Build `directory`: each name from `names` as a key, with the matching email from `emails` as its value.

c) Build `temps_c`: the same days as `temps_f`, converted to Celsius and rounded to 1 decimal. The formula is `(f - 32) * 5 / 9`.

d) Build `score_range`: for each student in `quiz_scores`, their highest score minus their lowest score. Use `np.max` and `np.min`.

In [0]:
# Your code here.
in_stock = {item: quantity for item, quantity in inventory.items() if quantity > 0}
print(in_stock)

directory = {names: emails for names, emails in zip(names, emails)}
print(directory)

temps_c = {day: round((temp - 32) * 5 / 9, 1) for day, temp in temps_f.items()}
print(temps_c)

score_range = {name: int(np.max(scores) - np.min(scores)) for name, scores in quiz_scores.items()}
print(score_range)

<details>
<summary><b>Show answer</b></summary>

```python
# a) a filter, so the if goes at the end; .items() gives you the count to test
in_stock = {item: count for item, count in inventory.items() if count > 0}
# {'pens': 12, 'markers': 5}

# b) two lists side by side -> zip
directory = {name: email for name, email in zip(names, emails)}
# {'Ana': 'ana@example.com', 'Ben': 'ben@example.com', 'Chloe': 'chloe@example.com'}

# c) same keys, transformed values
temps_c = {day: round((f - 32) * 5 / 9, 1) for day, f in temps_f.items()}
# {'Mon': 20.0, 'Tue': 23.9, 'Wed': 15.0}

# d) the value can be any expression, including NumPy calls
score_range = {name: int(np.max(scores) - np.min(scores)) for name, scores in quiz_scores.items()}
# {'Ana': 16, 'Ben': 15, 'Chloe': 8}
```

If you got stuck, write the loop first (`new_dict = {}`, then `for ...:`, then `new_dict[key] = value`) and translate it with the recipe in 3.4.

</details>

---
# 4. `lambda` and `map()`

**From Matei's post:** You want more practice with `map()` and `lambda`. You could follow the examples, but you are not confident writing them on your own yet.

These two feel slippery because they rely on one idea the course moves past quickly: **a function is a value**, just like a number or a list. You can store it in a variable, or hand it to another function. Once that idea is solid, `lambda` is just a shorter way to write a function, and `map()` is a function that takes another function as its input.

### 4.1 A function without parentheses is the function itself

In [0]:
def double(x):
    return x * 2

print(double(5))    # with parentheses: CALL the function and get its result
print(double)       # without parentheses: the function itself

In [0]:
twice = double      # a second name for the same function (no parentheses, so nothing is called)

print(twice(7))

Hold on to this difference: `double(5)` is a **result** (`10`). `double` is the **function**, something you can pass along so that another piece of code can call it later.

### 4.2 `lambda` is a one-line function without a name

Any function whose body is a single `return` line can be written as a `lambda`:

```python
def NAME(PARAMETERS):
    return EXPRESSION
```

becomes

```python
lambda PARAMETERS: EXPRESSION
```

**The recipe:** delete `def` and the name, write `lambda`, drop the parentheses around the parameters, delete `return`, and keep it all on one line.

In [0]:
# def version
def square(n):
    return n ** 2

# lambda version (stored in a name here only so we can compare the two)
square_lambda = lambda n: n ** 2

print(square(6), square_lambda(6))

More translations. Cover the right column and try writing each `lambda` yourself first.

| `def` version | `lambda` version |
|---|---|
| `def add(a, b): return a + b` | `lambda a, b: a + b` |
| `def is_even(n): return n % 2 == 0` | `lambda n: n % 2 == 0` |
| `def shout(word): return word.upper() + "!"` | `lambda word: word.upper() + "!"` |
| `def label(s): return "pass" if s >= 70 else "fail"` | `lambda s: "pass" if s >= 70 else "fail"` |
| `def full_name(first, last): return f"{last}, {first}"` | `lambda first, last: f"{last}, {first}"` |

In [0]:
shout = lambda word: word.upper() + "!"
print(shout("hello"))

full_name = lambda first, last: f"{last}, {first}"
print(full_name("Ada", "Lovelace"))

What a `lambda` **cannot** contain: the word `return`, an assignment with `=`, or more than one statement. It is exactly one expression. If you need more than that, write a regular `def`.

Storing a lambda in a variable, as in the cells above, is fine for practice, but in real code that is what `def` is for. Lambdas earn their place when you hand them **directly** to another function, which is the rest of this section.

### 4.3 `map(function, items)` calls a function on every item

In [0]:
celsius = [0, 12.5, 25, 37, 100]

fahrenheit = map(lambda c: c * 9 / 5 + 32, celsius)
print(fahrenheit)

That `<map object at 0x...>` is not an error. `map` is **lazy**: it does not compute anything until something asks for the values. `list()` asks for all of them:

In [0]:
fahrenheit = map(lambda c: c * 9 / 5 + 32, celsius)

print(list(fahrenheit))

A map object can only be walked through **once**. After `list()` has used it up, it is empty. **Predict** the second line before you run this:

In [0]:
fahrenheit = map(lambda c: c * 9 / 5 + 32, celsius)

print(list(fahrenheit))
print(list(fahrenheit))

One habit avoids both surprises: **wrap `map(...)` in `list(...)` right away**, and store the list.

In [0]:
fahrenheit = list(map(lambda c: c * 9 / 5 + 32, celsius))

print(fahrenheit)
print(fahrenheit)    # a list can be used as many times as you like

### 4.4 Three ways to write the same thing

`map` does not need a `lambda`. It accepts **any** function, as long as you pass the function itself, without parentheses.

In [0]:
def to_fahrenheit(c):
    return c * 9 / 5 + 32

with_def = list(map(to_fahrenheit, celsius))                  # a named function: no parentheses!
with_lambda = list(map(lambda c: c * 9 / 5 + 32, celsius))    # a lambda
with_comprehension = [c * 9 / 5 + 32 for c in celsius]        # a list comprehension (Section 2)

print(with_def)
print(with_def == with_lambda == with_comprehension)

In [0]:
with_comprehension

Adding parentheses is the most common `map` mistake. `to_fahrenheit()` tries to **call** the function right now, with no temperature, instead of handing the function to `map` so that `map` can call it once per item:

In [0]:
# SUPPOSED TO FAIL -- TypeError
list(map(to_fahrenheit(), celsius))

`map` is at its best with functions that already exist, like the built-ins `int`, `len`, and `str.upper`. Converting a list of numbers stored as text is a very common real use:

In [0]:
typed_in = ["4", "8", "15", "16", "23", "42"]

as_numbers = list(map(int, typed_in))
print(as_numbers)
print(sum(as_numbers))

words = ["map", "lambda", "filter"]
print(list(map(len, words)))
print(list(map(str.upper, words)))

`map` can also walk through **two lists at once**, passing one item from each to a function with two parameters. Compare this with the `zip` comprehension in 2.5:

In [0]:
unit_prices = [2.50, 10.00, 4.25]
quantities = [4, 1, 3]

print(list(map(lambda price, qty: price * qty, unit_prices, quantities)))

### 4.5 Where `lambda` really earns its keep: `key=`

`sorted`, `max`, and `min` accept a `key=` argument: **a function that receives one item and returns the thing to compare.** A `lambda` lets you write that function right where you need it.

In [0]:
team = [("Ana", 88), ("Ben", 72), ("Chloe", 95), ("Dev", 81)]

print(sorted(team))                                               # default: alphabetical by name
print(sorted(team, key=lambda person: person[1]))                 # by score
print(sorted(team, key=lambda person: person[1], reverse=True))   # by score, highest first
print(max(team, key=lambda person: person[1]))                    # the highest score

Read `key=lambda person: person[1]` as: "to compare two people, look at item `[1]` of each one, the score."

The same idea works on a dictionary's `.items()` pairs from Section 3:

In [0]:
stock = {"pens": 12, "markers": 5, "staplers": 30}

print(sorted(stock.items(), key=lambda pair: pair[1]))    # (item, count) pairs, by count
print(max(stock, key=lambda item: stock[item]))           # the key with the biggest count

Finally, `filter(function, items)` keeps the items for which the function returns `True`. Like `map`, it is lazy, so wrap it in `list()`:

In [0]:
numbers = [3, 8, 11, 14, 20, 7]

print(list(filter(lambda n: n % 2 == 0, numbers)))
print([n for n in numbers if n % 2 == 0])    # the comprehension version, for comparison

### 4.6 How to write one yourself

When a problem calls for `map` or `key=`, walk through these steps, out loud if it helps:

1. **Say what should happen to ONE item.** "Take a name and make it all capitals."
2. **Write that as an expression, using a placeholder name for the item.** `name.upper()`
3. **Put `lambda name:` in front of it.** `lambda name: name.upper()`
4. **Test it on one value before using it on a whole list.**
5. **Hand it to `map` (or `key=`), and wrap `map` in `list()`.**

In [0]:
names = ["ana", "ben", "chloe"]

make_caps = lambda name: name.upper()
print(make_caps("ana"))               # step 4: test on one value

print(list(map(make_caps, names)))    # step 5: then the whole list

**So which should you use, `map` or a comprehension?** For new code, most Python programmers reach for a comprehension, and it is fine if you do too. You still need to be able to **read** `map` and `lambda`, because the course and other people's code use them, and `key=lambda ...` has no comprehension equivalent.

#### Your Turn 4.1 - Write the lambda

Run this cell first. For each part, test your lambda on a single value before you use it on a list (step 4).

In [0]:
words = ["python", "notebook", "kernel", "module"]
amounts = [19.5, 5, 120.2]
typed_scores = ["88", "92", "79"]
products = [("laptop", 999.99), ("mouse", 24.50), ("monitor", 189.00)]

a) Write a `lambda` that returns the **last letter** of a word. Use it with `map` on `words`.

b) Use `map` and a `lambda` to turn each number in `amounts` into a dollar string like `"$19.50"`. Hint: `f"${x:.2f}"`.

c) Use `map` with a built-in function (no lambda needed) to turn `typed_scores` into integers, then compute their average.

d) Sort `words` by length, shortest first.

e) Find the product in `products` with the **highest** price.

f) Rewrite `list(map(lambda w: w[::-1], words))` as a list comprehension.

In [0]:
# Your code here.
print('*** a ***')
print('testing'[-1])
last_letters = list(map(lambda word: word[-1], words))
print(list(last_letters))
print('*** b ***')
print(f'${19:.2f}')
dollars = list(map(lambda amount: f"${amount:.2f}", amounts))
print(dollars)
print('*** c ***')
print(int(19.50))
int_scores = list(map(int, typed_scores))
print(sum(int_scores) / len(int_scores))
print('*** d ***')
print(len('testing)'))
print(sorted(words, key=len))
print('*** e ***')
print(list(sorted(products, key=lambda product: product[1]))[-1])
print('*** f ***')
#[EXPRESSION for ITEM in ITERABLE if CONDITION]
print('word'[::-1])
print([w[::-1] for w in words])
